# Blackjack Reinforcement Learning Experiment
In this notebook, we train a Reinforcement Learning agent using **Tabular Q-Learning** to play Blackjack. The agent interacts with the custom environment defined in the `blackjack.py` file.

## The Deck & The Rules

* The game uses a standard French playing card deck (or multiple combined decks in extended mode).
* Number cards count as their face value, face cards count as 10, and Aces can count as either 11 or 1.
* **Objective:** Achieve a higher score than the dealer without exceeding 21 points.

## Import Fail Troubleshooting

In [1]:
import os
os.chdir(r'C:\Users\freese\Desktop\Allgemein\GitLab\ml-blackjack')

## 1. Setup & Hyperparameters
First, we import the necessary libraries as well as our custom environment from the `blackjack.py` file. We also define the hyperparameters required for the Q-Learning algorithm.

In [2]:
import numpy as np
import collections
import random
from blackjack import BlackjackEnv  # Imports your class from blackjack.py

# Agent Hyperparameters
alpha = 0.01      # Learning rate (How much new information overrides old knowledge)
gamma = 1.0       # Discount factor (1.0 = Full focus on the end of the round)
epsilon = 0.1     # Exploration rate (10% probability of choosing a random move)
num_episodes = 500000

# Initialize Q-Table
# Index 0: Hit, Index 1: Stand (Aligned with the step() method in the environment)
Q = collections.defaultdict(lambda: np.zeros(2))

def choose_action(state, epsilon):
    """Epsilon-Greedy policy for action selection"""
    if random.random() < epsilon:
        return random.choice([0, 1])  # Exploration (Random move)
    else:
        return np.argmax(Q[state])     # Exploitation (Best known move)

## 2. The Training Loop (Q-Learning)
We let the agent play through the specified number of episodes. After each action, we update the Q-table based on the received reward using the Bellman optimality equation:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left( r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right)$$

In [ ]:
# Start the environment in extended mode
env = BlackjackEnv(state_mode="extended")

# Track statistics for performance evaluation
win_loss_draw = [0, 0, 0] 

for episode in range(num_episodes):
    state = env.start_round()
    done = False
    
    while not done:
        # 1. Select action (0=Hit, 1=Stand)
        action = choose_action(state, epsilon)
        
        # 2. Execute step in the environment
        next_state, reward, done = env.step(action)
            
        # 3. Determine the best theoretical next action for the update
        best_next_action = np.argmax(Q[next_state])
        
        # Bellman Update (If done=True, the future expected value drops out)
        td_target = reward + gamma * Q[next_state][best_next_action] * (not done)
        td_delta = td_target - Q[state][action]
        Q[state][action] += alpha * td_delta
        
        # State transition for the next loop iteration
        state = next_state
        
    # Record statistics at the end of each round
    if reward == 1:
        win_loss_draw[0] += 1
    elif reward == -1:
        win_loss_draw[1] += 1
    else:
        win_loss_draw[2] += 1

print(f"Training finished after {num_episodes} episodes.")
print(f"Wins: {win_loss_draw[0]} | Losses: {win_loss_draw[1]} | Draws: {win_loss_draw[2]}")
print(f"Win Rate during training: {win_loss_draw[0] / num_episodes * 100:.2f}%")

## 3. Evaluating the Trained Agent
To evaluate how well the learned strategy performs, we test the agent over 10,000 separate rounds. Here, exploration is completely disabled (equivalent to $\epsilon = 0$), so the agent acts strictly based on the highest expected utility in its Q-table.

In [ ]:
test_episodes = 10000
test_wins = 0
test_losses = 0
test_draws = 0

for _ in range(test_episodes):
    state = env.start_round()
    done = False
    
    while not done:
        # Greedily choose the best action from the Q-table (no randomness!)
        action = np.argmax(Q[state])
        state, reward, done = env.step(action)
        
    if reward == 1:
        test_wins += 1
    elif reward == -1:
        test_losses += 1
    else:
        test_draws += 1

print(f"Testing phase completed over {test_episodes} rounds.")
print(f"Wins: {test_wins} | Losses: {test_losses} | Draws: {test_draws}")
print(f"Final Win Rate: {test_wins / test_episodes * 100:.2f}%")